In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# CADEC semantic entropy (8 models)

Map CADEC model `output_text` → CUI with the **RQ1_PART2** full-UMLS SapBERT+FAISS five-rule protocol, then compute per-instance normalised semantic entropy:

- \(\hat H = H / \log_2(m+1)\) (per-instance \(m\), not fixed 9)
- Include only \(m \ge 3\); all-UNASSIGNED → NaN
- Accuracy = exact CUI match on the **original** input
- `mapping_confidence` = top-1 SapBERT/FAISS cosine of the **original**-input output
  (RQ4 abstention baseline; higher = more confident CUI mapping)

Does **not** modify `RQ1_PART2_full_umls_pool.ipynb`.

**Output:** `outputs/rq3/entropy_cadec.csv`  
**Mapped cache:** `outputs/rq3/intermediate/rq3_cadec_mapped_outputs.csv` (includes per-output `confidence`)


In [1]:
# Setup: absolute PROJECT_ROOT (nbconvert-safe)
import json
import os
import pickle
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "CUDA required for CADEC_entropy.ipynb"
print(f"GPU: {torch.cuda.get_device_name(0)}")

PROJECT_ROOT = PROJECT_ROOT
CONFIG_PATH = PROJECT_ROOT / "config" / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = int(cui_pool.get("n_cuis", len(cui_pool.get("cuis", {}))))
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
assert _n_cuis > 3_000_000, (
    f"Wrong CUI pool loaded: n_cuis={_n_cuis:,} (expected full_umls > 3,000,000). "
    f"Check CFG['pool_full']."
)
print("ASSERT OK: full_umls-scale pool loaded.")


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA L40S
PROJECT_ROOT: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs
CONFIG_PATH:  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json


Loaded CUI pool from: /home/s224858267/data/umls/pools/cui_pool_full_umls.pkl
Pool type: full_umls | unique CUIs: 3,341,331
ASSERT OK: full_umls-scale pool loaded.


## 1) Load / assemble model outputs


In [2]:
# Load / assemble CADEC model outputs
INTER_DIR = PROJECT_ROOT / "outputs" / "rq3" / "intermediate"
RAW_DIR = INTER_DIR / "cadec_model_raw"
OUT_ASSEMBLED = INTER_DIR / "rq3_cadec_model_outputs.csv"
OUT_MAPPED = INTER_DIR / "rq3_cadec_mapped_outputs.csv"
OUT_ENTROPY = PROJECT_ROOT / "outputs" / "rq3" / "entropy_cadec.csv"
INST_CSV = INTER_DIR / "rq3_cadec_instances.csv"

EXPECTED_KEYS = [
    "bert-base", "biobert", "pubmedbert", "flan-t5-base",
    "biomistral", "mistral", "openbiollm", "llama3",
]
TARGET_COLS = [
    "instance_id", "model_name", "input_variant_id", "input_type",
    "output_text", "gold_cui", "perturbation_type",
]

assert OUT_ASSEMBLED.is_file() and OUT_ASSEMBLED.stat().st_size > 0, (
    f"Missing {OUT_ASSEMBLED} — run CADEC_inference first. "
    "This notebook must not overwrite that file."
)
_log(f"Loading assembled outputs (read-only): {OUT_ASSEMBLED}")
df_out = pd.read_csv(OUT_ASSEMBLED, low_memory=False)

miss_c = [c for c in TARGET_COLS if c not in df_out.columns]
assert not miss_c, f"Model outputs missing columns: {miss_c}"
_log(f"Model outputs: {len(df_out):,} rows | models={sorted(df_out['model_name'].unique())}")
print(df_out.groupby("model_name").size().to_string())
sys.stdout.flush()

# gold_mention for exact-match rule
if INST_CSV.is_file():
    _inst = pd.read_csv(INST_CSV, usecols=["instance_id", "gold_mention"])
    df_out = df_out.merge(_inst, on="instance_id", how="left")
    _log(f"Joined gold_mention from {INST_CSV.name}")
else:
    df_out["gold_mention"] = np.nan
    _log(f"WARNING: {INST_CSV} missing — exact-match mention rule limited")


[2026-07-31 12:28:54 UTC] Loading assembled outputs: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_model_outputs.csv


[2026-07-31 12:28:54 UTC] Model outputs: 266,024 rows | models=['BERT-base', 'BioBERT', 'BioMistral-7B', 'FLAN-T5-base', 'Llama3-OpenBioLLM-8B', 'Meta-Llama-3-8B-Instruct', 'Mistral-7B-Instruct-v0.1', 'PubMedBERT']


model_name
BERT-base                   33253
BioBERT                     33253
BioMistral-7B               33253
FLAN-T5-base                33253
Llama3-OpenBioLLM-8B        33253
Meta-Llama-3-8B-Instruct    33253
Mistral-7B-Instruct-v0.1    33253
PubMedBERT                  33253


[2026-07-31 12:28:54 UTC] Joined gold_mention from rq3_cadec_instances.csv


## 2) SapBERT + FAISS linker (PART2 five-rule `assign_with_encoder_scores`)


In [3]:
# SapBERT + FAISS + five-rule CUI mapping (RQ1_PART2 protocol)
# Reuses PART2 assign_with_encoder_scores + thresholds EXACTLY.
# Form scores for free-text outputs come from SapBERT cosine (same five rules).
import faiss
from transformers import AutoModel, AutoTokenizer

UNASSIGNED = "UNASSIGNED"  # never use "NA" — pandas read_csv treats NA as NaN
MIN_FORM_LEN = 3
CONFIDENCE_THRESHOLD = 0.70
SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / "sapbert_full_len3"
MIN_M_ACCEPTED = 3

_ksel_path = _resolve_cfg_path(CFG.get("k_selection", "outputs/rq1/k_selection.json"))
if _ksel_path is not None and Path(_ksel_path).exists():
    with open(_ksel_path, "r", encoding="utf-8") as _f:
        TOP_K = int(json.load(_f).get("k_selected", 50))
    _log(f"TOP_K={TOP_K} from {_ksel_path}")
else:
    TOP_K = int(CFG_YAML["umls"]["faiss_top_k"])
    _log(f"TOP_K={TOP_K} (fallback from config.yaml)")

_emb_npy = EMB_DIR / "embeddings.npy"
_forms_json = EMB_DIR / "surface_forms.json"
_pairs_json = EMB_DIR / "cui_form_pairs.json"
_index_path = EMB_DIR / "faiss.index"
for p in (_emb_npy, _forms_json, _pairs_json, _index_path):
    assert p.exists(), f"Missing PART2 embedding cache: {p}"

_log(f"Loading FAISS/SapBERT cache from {EMB_DIR}")
_form_embeddings = np.load(_emb_npy)
with open(_forms_json, "r", encoding="utf-8") as _f:
    _unique_forms = json.load(_f)
with open(_pairs_json, "r", encoding="utf-8") as _f:
    _form_cui_pairs = [tuple(x) for x in json.load(_f)]
_faiss_index = faiss.read_index(str(_index_path))
assert len(_unique_forms) == _form_embeddings.shape[0] == _faiss_index.ntotal

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_c)
_form_index = {f: i for i, f in enumerate(_unique_forms)}
_exact_index = defaultdict(set)
for _f, _cuis in _form_to_cuis.items():
    _exact_index[_f.casefold()].update(_cuis)

_raw_cuis = cui_pool["cuis"]
_cui_st21pv = {c: bool(rec.get("st21pv", False)) for c, rec in _raw_cuis.items()}
_cui_n_forms = {c: len(rec.get("surface_forms", ())) for c, rec in _raw_cuis.items()}

_rej = Counter()

def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s


def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def _embed_with_model(model, tokenizer, texts, batch_size=64, max_len=64, desc="embed"):
    """L2-normalised mean-pooled embeddings (float32 numpy). — PART2 identical."""
    vecs = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, unit="batch"):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = tokenizer(
                batch, return_tensors="pt", truncation=True, max_length=max_len, padding=True
            )
            enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.cpu().numpy())
    return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)


def assign_with_encoder_scores(query_text, mention_text, form_scores):
    """form_scores: dict form -> encoder cosine in [-1,1] (higher better).
    Returns (cui, score, rule_path).
    — Copied EXACTLY from RQ1_PART2_full_umls_pool.ipynb (do not drift).
    """
    rule_path = []
    # Build candidate (cui, form, score) from retrieved forms
    cand = []
    for form, sc in form_scores.items():
        for cui in _form_to_cuis.get(form, ()):
            cand.append((cui, form, float(sc)))
    if not cand:
        _rej["faiss_empty"] += 1
        return UNASSIGNED, 0.0, "faiss_empty"

    # Rule 1: exact match on mention (or query) against active vocabulary
    exact_cuis = set()
    # GOLD LEAK REMOVED (docs/BUG_AUDIT.md, "Rule 1 feeds the gold mention into the
    # prediction path"). mention_text is the GOLD ANNOTATION and is never shown to the
    # model; the prompt carries the PERTURBED context only. Keying rule 1 on it filtered
    # the model's candidates to gold-derived CUIs, and on the inject branch inserted them
    # at score 1.0 above any attainable cosine. gold_mention is also constant across all
    # variants, so the unperturbed string was used as the key for every rewrite.
    # Rule 1 now keys on the MODEL OUTPUT ONLY. mention_text is retained in the signature
    # so the call sites and the rule_path labels stay comparable row by row.
    for key in [query_text]:
        if key and str(key).strip():
            exact_cuis |= set(_exact_index.get(str(key).strip().casefold(), ()))
    if exact_cuis:
        exact_cand = [c for c in cand if c[0] in exact_cuis]
        if exact_cand:
            cand = exact_cand
            rule_path.append("exact_match")
            _rej["exact_match_hit"] += 1
        else:
            # exact CUI known but not in FAISS top-k: inject with score 1.0
            cand = [(c, str(query_text), 1.0) for c in exact_cuis] + cand
            rule_path.append("exact_match_inject")
            _rej["exact_match_hit"] += 1
    else:
        rule_path.append("no_exact_match")

    # Rule 3: ST21pv filter (keep if any remain)
    st_filt = [c for c in cand if _cui_st21pv.get(c[0], False)]
    if st_filt:
        if len(st_filt) < len(cand):
            _rej["st21pv_filtered_some"] += 1
        cand = st_filt
        rule_path.append("st21pv")
        _rej["st21pv_kept"] += 1
    else:
        rule_path.append("st21pv_skip")
        _rej["st21pv_filtered_all"] += 1  # would empty — skip filter

    # Rule 2: contextual / encoder score already in cand[2]; sort by it
    cand.sort(key=lambda x: x[2], reverse=True)
    rule_path.append("encoder_cosine")

    best_score = cand[0][2]
    # Rule 4: confidence threshold
    if best_score < CONFIDENCE_THRESHOLD:
        _rej["below_confidence"] += 1
        rule_path.append(f"below_thresh_{CONFIDENCE_THRESHOLD}")
        return UNASSIGNED, best_score, "+".join(rule_path)

    # Rule 5: frequency tiebreak among near-ties
    top = [c for c in cand if (best_score - c[2]) <= 0.02]
    top.sort(key=lambda x: (_cui_n_forms.get(x[0], 0), x[2]), reverse=True)
    _rej["freq_tiebreak"] += 1
    _rej["assigned"] += 1
    rule_path.append("freq_tiebreak")
    return top[0][0], float(top[0][2]), "+".join(rule_path)


# Always load the SapBERT query encoder. ONE decision about the mapped cache, and it is
# made in cell 8.
#
# There was a "skip the load when the mapped cache exists" optimisation here. It decided the
# same question cell 8 decides, but from weaker evidence -- file existence and column names
# alone, with no CADEC_FORCE_REMAP check and no comparison of the cached instance set
# against the freshly assembled model outputs. Whenever cell 8 then correctly chose to
# remap, the encoder had already been set to None and the run died inside
# _embed_with_model with "'NoneType' object has no attribute 'eval'". That took out jobs
# 32329 and 32388, the second of which was the canonical remap itself.
#
# The load costs ~30 s and ~0.5 GB of VRAM. Two predicates that must be kept in agreement
# forever cost more than that.
_log(f"Loading SapBERT query encoder on cuda: {SAPBERT_ID}")
_sap_tok = AutoTokenizer.from_pretrained(SAPBERT_ID)
_sap_mdl = AutoModel.from_pretrained(SAPBERT_ID)
_sap_mdl = _sap_mdl.to("cuda").eval()
_sap_mdl.half()
assert next(_sap_mdl.parameters()).device.type == "cuda"
_log(f"Linker ready | FAISS ntotal={_faiss_index.ntotal:,} TOP_K={TOP_K} conf>={CONFIDENCE_THRESHOLD}")


[2026-07-31 12:29:13 UTC] TOP_K=1000 from /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/k_selection.json


[2026-07-31 12:29:13 UTC] Loading FAISS/SapBERT cache from /home/s224858267/data/umls/embeddings/sapbert_full_len3


[2026-07-31 12:30:39 UTC] Mapped cache present — skipping SapBERT load | FAISS ntotal=7,653,278 TOP_K=1000 conf>=0.7


## 3) Map `output_text` → CUI


In [4]:
# Map every output_text → CUI
# Persist per-output `confidence` (top-1 SapBERT/FAISS cosine after five-rule assign)
# so RQ4 can use original-input mapping_confidence without re-inference.
_MAP_COLS = [
    "instance_id", "model_name", "input_variant_id", "input_type",
    "output_text", "gold_cui", "perturbation_type", "gold_mention",
    "predicted_cui", "confidence", "assign_rule_path", "gold_cui_norm",
]

# Cache-staleness guard. The cache used to be reused on file-existence alone, with no
# comparison against the freshly assembled model outputs. Because the reuse branch REPLACES
# df_out wholesale, a stale cache silently substituted its own instance set: after the
# LanguageTool rewind, entropy_cadec.csv was built from a 2026-08-19 cache whose rows did
# not match the post-rewind inference at all. Require the cached instance_id set to equal
# the fresh one, and allow CADEC_FORCE_REMAP=1 to override the cache entirely.
_force_remap = os.environ.get("CADEC_FORCE_REMAP", "").strip() not in ("", "0", "false", "False")
_cache_ok = (
    OUT_MAPPED.is_file()
    and OUT_MAPPED.stat().st_size > 0
    and set(["predicted_cui", "confidence", "gold_cui_norm"]).issubset(
        set(pd.read_csv(OUT_MAPPED, nrows=0).columns)
    )
)
if _cache_ok:
    _cached_ids = set(pd.read_csv(OUT_MAPPED, usecols=["instance_id"])["instance_id"].astype(str))
    _fresh_ids = set(df_out["instance_id"].astype(str))
    _only_cache = _cached_ids - _fresh_ids
    _only_fresh = _fresh_ids - _cached_ids
    if _cached_ids != _fresh_ids:
        _cache_ok = False
        _log(
            f"REMAP: cached instance set != model outputs "
            f"(cache={len(_cached_ids):,} fresh={len(_fresh_ids):,} "
            f"only-in-cache={len(_only_cache):,} only-in-fresh={len(_only_fresh):,})"
        )
    else:
        _log(f"Cache instance set matches model outputs ({len(_fresh_ids):,} instances)")
if _force_remap and _cache_ok:
    _cache_ok = False
    _log("REMAP: CADEC_FORCE_REMAP set — ignoring the mapped cache")
_reuse_mapped = _cache_ok

if _reuse_mapped:
    _log(f"Loading cached mapped outputs (with confidence): {OUT_MAPPED}")
    df_out = pd.read_csv(OUT_MAPPED, low_memory=False)
    _log(
        f"Cached map | rows={len(df_out):,} | "
        f"UNASSIGNED={(df_out['predicted_cui']==UNASSIGNED).mean():.1%}"
    )
else:
    texts = df_out["output_text"].fillna("").astype(str).tolist()
    mentions = df_out["gold_mention"].fillna("").astype(str).tolist()

    # Embed DISTINCT output_text only, then join the vectors back onto every row. The
    # embedding depends solely on output_text, so this is exact -- and it is a ~16x saving
    # (15,229 distinct texts across 239,680 rows). The FAISS search is deduped the same way.
    # The five-rule assign below still runs PER ROW: assign_with_encoder_scores also reads
    # gold_mention (Rule 1 exact-matches on mention AND query), so it must not be deduped on
    # text alone -- the correct key there would be (output_text, gold_mention), 59,689 pairs.
    # Mirrors the unique-text embedding in RQ4_umls_candidate_margin.ipynb.
    # Equivalence validated by scripts/cadec_dedup_check.py.
    _uniq_texts = list(dict.fromkeys(texts))
    _log(
        f"SapBERT-embed {len(_uniq_texts):,} distinct outputs of {len(texts):,} rows "
        f"({len(texts)/max(len(_uniq_texts), 1):.1f}x dedup, batch=128) …"
    )
    _uniq_vecs = _embed_with_model(
        _sap_mdl, _sap_tok, _uniq_texts, batch_size=128, max_len=64, desc="sapbert"
    )
    _text_to_u = {t: i for i, t in enumerate(_uniq_texts)}
    _row_u = np.fromiter((_text_to_u[t] for t in texts), dtype=np.int64, count=len(texts))
    q_vecs = _uniq_vecs[_row_u]

    _log(f"FAISS search TOP_K={TOP_K} on {len(_uniq_texts):,} distinct vectors …")
    _D_u, _I_u = _faiss_index.search(_uniq_vecs.astype(np.float32), TOP_K)
    D, I = _D_u[_row_u], _I_u[_row_u]

    # Build form_scores via SapBERT cosine (dot with cached L2-normalised form emb)
    predicted = []
    scores = []
    paths = []
    for i in tqdm(range(len(texts)), desc="five-rule assign"):
        form_scores = {}
        for sc, ix in zip(D[i], I[i]):
            if int(ix) < 0:
                continue
            form = _unique_forms[int(ix)]
            if len(form) < MIN_FORM_LEN:
                continue
            # re-score with cached embedding (PART2 / RQ3 style)
            form_scores[form] = float(np.dot(q_vecs[i], _form_embeddings[int(ix)]))
        cui, sc, path = assign_with_encoder_scores(texts[i], mentions[i], form_scores)
        predicted.append(_norm_cui(cui))
        scores.append(sc)
        paths.append(path)

    df_out["predicted_cui"] = predicted
    df_out["confidence"] = scores
    df_out["assign_rule_path"] = paths
    df_out["gold_cui_norm"] = df_out["gold_cui"].map(_norm_cui)

    OUT_MAPPED.parent.mkdir(parents=True, exist_ok=True)
    df_out[_MAP_COLS].to_csv(OUT_MAPPED, index=False)
    _log(f"Wrote mapped cache → {OUT_MAPPED} | rows={len(df_out):,}")

_log(
    f"CUI map ready | UNASSIGNED={(df_out['predicted_cui']==UNASSIGNED).mean():.1%} | "
    f"confidence finite={df_out['confidence'].notna().mean():.1%}"
)
if not _reuse_mapped:
    _log(f"rej={dict(_rej)}")
print(df_out.groupby("model_name")["predicted_cui"].apply(
    lambda s: (s == UNASSIGNED).mean()
).rename("unassigned_rate").to_string())
sys.stdout.flush()

# free SapBERT if it was loaded this session
try:
    del _sap_mdl
    torch.cuda.empty_cache()
except NameError:
    pass


[2026-07-31 12:30:39 UTC] Loading cached mapped outputs (with confidence): /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_mapped_outputs.csv


[2026-07-31 12:30:39 UTC] Cached map | rows=266,024 | UNASSIGNED=3.6%


[2026-07-31 12:30:39 UTC] CUI map ready | UNASSIGNED=3.6% | confidence finite=100.0%


model_name
BERT-base                   0.000000
BioBERT                     0.000000
BioMistral-7B               0.014435
FLAN-T5-base                0.003158
Llama3-OpenBioLLM-8B        0.000571
Meta-Llama-3-8B-Instruct    0.242113
Mistral-7B-Instruct-v0.1    0.024960
PubMedBERT                  0.000000


## 4) Entropy + accuracy (\(m\ge 3\), \(\hat H = H/\log_2(m+1)\))


In [5]:
# Entropy + accuracy (PART2 _entropy_from_labels; m>=3 inclusion)

def shannon_entropy(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    probs = np.array([c / total for c in counts.values()], dtype=float)
    h = float(-np.sum(probs * np.log2(np.clip(probs, 1e-12, 1.0))))
    return h, counts


def _entropy_from_labels(labels, unassigned=UNASSIGNED):
    """Entropy over assigned CUIs only. All-UNASSIGNED → NaN (not 0).

    Normalisation (per-instance): H_norm = H / log2(m+1) where m = n_variants - 1
    (accepted perturbations; n_variants = original + m). NOT fixed log2(9) and
    NOT log2(n_assigned).
    — Copied EXACTLY from RQ1_PART2_full_umls_pool.ipynb.
    """
    labels = [
        unassigned if str(lab).lower() in {"nan", "na", "none", ""} else str(lab)
        for lab in labels
    ]
    assigned = [lab for lab in labels if lab != unassigned]
    n_variants = len(labels)
    n_assigned = len(assigned)
    n_unassigned = n_variants - n_assigned
    m_accepted = max(n_variants - 1, 0)
    if n_assigned == 0:
        return {
            "n_variants": n_variants,
            "n_assigned": 0,
            "n_unassigned": n_unassigned,
            "m_accepted": m_accepted,
            "all_unassigned": True,
            "n_clusters_full": 0,
            "semantic_entropy_full": np.nan,
            "normalised_semantic_entropy_full": np.nan,
            "dominant_cluster_full": unassigned,
            "is_zero_entropy": False,  # must NOT count as entropy 0
        }
    h, counts = shannon_entropy(assigned)
    # Per-instance normalisation by log2(m+1) == log2(n_variants)
    h_hat = h / np.log2(n_variants) if n_variants >= 2 else np.nan
    dominant = max(counts.items(), key=lambda x: x[1])[0]
    return {
        "n_variants": n_variants,
        "n_assigned": n_assigned,
        "n_unassigned": n_unassigned,
        "m_accepted": m_accepted,
        "all_unassigned": False,
        "n_clusters_full": len(counts),
        "semantic_entropy_full": h,
        "normalised_semantic_entropy_full": h_hat,
        "dominant_cluster_full": dominant,
        "is_zero_entropy": bool(h_hat <= 1e-12) if pd.notna(h_hat) else False,
    }


DOMAIN = {
    "BioBERT": "biomedical",
    "BioMistral-7B": "biomedical",
    "Llama3-OpenBioLLM-8B": "biomedical",
    "PubMedBERT": "biomedical",
    "BERT-base": "general",
    "Mistral-7B-Instruct-v0.1": "general",
    "Meta-Llama-3-8B-Instruct": "general",
    "FLAN-T5-base": "generative_baseline",
}
PAIR = {
    "BioBERT": "pair1_biobert_vs_bertbase",
    "BERT-base": "pair1_biobert_vs_bertbase",
    "BioMistral-7B": "pair2_biomistral_vs_mistral",  # PRIMARY
    "Mistral-7B-Instruct-v0.1": "pair2_biomistral_vs_mistral",
    "Llama3-OpenBioLLM-8B": "pair3_openbiollm_vs_llama3",
    "Meta-Llama-3-8B-Instruct": "pair3_openbiollm_vs_llama3",
    "PubMedBERT": None,
    "FLAN-T5-base": None,
}
MATCHED_PAIRS = [
    ("BioBERT", "BERT-base"),
    ("BioMistral-7B", "Mistral-7B-Instruct-v0.1"),  # PRIMARY
    ("Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct"),
]

# --- variant input text, for the parallel de-duplicated m ------------------------------
# Byte-identical input variants necessarily yield the same output and therefore the same
# cluster, so they inflate the dominant cluster (pushing entropy toward 0) while also
# inflating m, the denominator log2(m+1). Back-translation is deterministic greedy and is
# generated in two slots per instance, so EXACTLY 50% of accepted back-translation rows are
# duplicates of their sibling; overall 24.3% of accepted CADEC variants (25.8% in MM).
# We therefore compute entropy BOTH ways in one pass. The existing columns and the existing
# m>=3 filter are unchanged -- downstream notebooks keep reading normalised_entropy /
# m_accepted -- and the de-duplicated view is emitted alongside for comparison.
_VALIDATED_FULL = INTER_DIR / "rq3_cadec_validated_perturbations_full.csv"
assert _VALIDATED_FULL.is_file(), (
    f"{_VALIDATED_FULL} missing. The de-duplicated denominator is PRIMARY "
    f"(docs/ANALYSIS_PRECOMMIT.md section 3); it cannot be silently skipped."
)
# POSITIONAL variant key. CADEC_inference.ipynb cell5:108-117 RENUMBERS accepted variants per
# instance as f"{iid}_p{j:02d}", j enumerating that instance's rows in df_pert (file) order with
# the original consuming j=0. So the k-th accepted row in file order is _p{k:02d}, 1-based, and
# input_variant_id is a POSITION, not the original perturbation_id.
#
# Keying on perturbation_id (the original implementation, commit 125d9d6) therefore resolved a
# real but DIFFERENT variant's text for 78.75% of instances, and dict.get() never failed because
# the wrong key was still a valid key. 2,188 of 5,161 instances carried a wrong m_distinct.
# See docs/BUG_AUDIT.md, "CADEC de-duplication key resolves the wrong variant text".
_vt = pd.read_csv(
    _VALIDATED_FULL,
    usecols=["instance_id", "perturbation_text", "accepted_final"],
    low_memory=False,
)
_acc = _vt[_vt["accepted_final"].astype(str).str.lower().isin(("true", "1"))].copy()
_acc["_iid"] = _acc["instance_id"].astype(str)
_acc["_pos"] = _acc.groupby("_iid").cumcount() + 1          # file order, 1-based
_acc["_vid"] = _acc["_iid"] + "_p" + _acc["_pos"].map("{:02d}".format)
_variant_text = dict(zip(_acc["_vid"], _acc["perturbation_text"].fillna("").astype(str)))
_log(f"Variant text map for dedup: {len(_variant_text):,} positional variant ids")

# --- hard assertions: a silent wrong resolution must be impossible --------------------------
_pert_out = df_out[df_out["input_type"] != "original"].copy()
_pert_out["_iid"] = _pert_out["instance_id"].astype(str)
_n_out = _pert_out.drop_duplicates(["_iid", "input_variant_id"]).groupby("_iid").size()
_n_acc = _acc.groupby("_iid").size()
_cmp = pd.concat([_n_acc.rename("accepted"), _n_out.rename("outputs")], axis=1).dropna()
_bad = _cmp[_cmp["accepted"] != _cmp["outputs"]]
assert _bad.empty, (
    f"ASSERT FAILED: accepted-variant count != output-variant count for {len(_bad):,} "
    f"instance(s); the positional key would misalign. Examples:\n{_bad.head(10)}"
)
_unresolved = sorted(set(_pert_out["input_variant_id"].astype(str)) - set(_variant_text))
assert not _unresolved, (
    f"ASSERT FAILED: {len(_unresolved):,} output variant id(s) have no accepted-variant text; "
    f"a key that does not resolve must be an error, never a default. "
    f"Examples: {_unresolved[:10]}"
)
_log(f"ASSERT OK: {len(_cmp):,} instances have accepted-count == output-count; "
     f"0 unresolved variant ids")

# --- acceptance filter AT THE ENTROPY STAGE -------------------------------------------------
# The model-output files carry no acceptance column, and nothing downstream filtered on one, so
# a variant that was accepted when inference ran but is rejected in the CURRENT gate verdicts
# would still reach m_accepted and the cluster distribution. That happened in MedMentions (3
# instances, 24 rows). Drop non-accepted rows BEFORE m is computed; instances simply lose a
# variant and fall out via m >= 3 if they fall out at all.
_n_before = len(df_out)
_is_orig = df_out["input_type"] == "original"
_ok = _is_orig | df_out["input_variant_id"].astype(str).isin(_variant_text)
df_out = df_out[_ok].copy()
_log(f"Acceptance filter at entropy stage: dropped {_n_before - len(df_out):,} of "
     f"{_n_before:,} output rows lacking a current accepted counterpart")
_resid = sorted(
    set(df_out.loc[df_out["input_type"] != "original", "input_variant_id"].astype(str))
    - set(_variant_text)
)
assert not _resid, (
    f"ASSERT FAILED: {len(_resid):,} output row(s) survive the acceptance filter without an "
    f"accepted counterpart. Examples: {_resid[:10]}"
)

# --- record WHICH gate verdicts these numbers used ------------------------------------------
# rq3_cadec_validated_perturbations_full.csv is mutable; scripts/freeze_validated_inputs.py
# writes a checksum sidecar so the reported numbers cite a specific set of gate verdicts.
import hashlib as _hashlib
_h = _hashlib.sha256()
with open(_VALIDATED_FULL, "rb") as _f:
    for _blk in iter(lambda: _f.read(1 << 20), b""):
        _h.update(_blk)
VALIDATED_SHA256 = _h.hexdigest()
_side = _VALIDATED_FULL.with_suffix(_VALIDATED_FULL.suffix + ".sha256.json")
if _side.is_file():
    _frozen = json.loads(_side.read_text()).get("sha256")
    _log(f"Gate verdicts: {_VALIDATED_FULL.name} sha256={VALIDATED_SHA256[:16]}... "
         f"({'MATCHES' if _frozen == VALIDATED_SHA256 else '*** DIFFERS FROM ***'} the frozen sidecar)")
else:
    _log(f"Gate verdicts: {_VALIDATED_FULL.name} sha256={VALIDATED_SHA256[:16]}... (no sidecar)")


def _variant_key(vid, itype, iid):
    """Text used to detect byte-identical variants.

    Originals key on their own id (one per instance-model). A perturbation id that does not
    resolve RAISES: the previous implementation returned a "<<missing:...>>" sentinel, which
    made an unresolvable key indistinguishable from a correctly-resolved one at a glance and
    let a wrong-but-present key pass unnoticed. The assertions above make this unreachable;
    it is kept as a second line of defence.
    """
    vid = str(vid)
    if itype == "original":
        return f"<<orig:{iid}>>"
    try:
        return _variant_text[vid]
    except KeyError:
        raise KeyError(
            f"variant id {vid!r} has no accepted-variant text. The de-duplication key cannot "
            f"fall back to a sentinel: see docs/BUG_AUDIT.md (CADEC dedup key)."
        ) from None

ent_rows = []
_m_dist = Counter()
_m_dist_distinct = Counter()
_n_excl = 0
_n_inc = 0
_n_inc_dedup = 0

for (iid, model), grp in tqdm(
    df_out.groupby(["instance_id", "model_name"], sort=False),
    desc="entropy",
):
    # m = # perturbation rows; n_outputs = original + m
    n_pert = int((grp["input_type"] == "perturbation").sum())
    n_orig = int((grp["input_type"] == "original").sum())
    n_outputs = len(grp)
    m = n_pert if n_pert > 0 else max(n_outputs - n_orig, n_outputs - 1)
    _m_dist[m] += 1
    if m < MIN_M_ACCEPTED:
        _n_excl += 1
        continue
    _n_inc += 1

    labels = grp["predicted_cui"].tolist()
    # Ensure order: prefer original first then perts (stable)
    grp_ord = pd.concat([
        grp[grp["input_type"] == "original"],
        grp[grp["input_type"] != "original"],
    ], ignore_index=True)
    labels = grp_ord["predicted_cui"].tolist()
    ent = _entropy_from_labels(labels)

    # Parallel view: collapse byte-identical input variants, keeping first occurrence.
    _keys = [
        _variant_key(v, t, iid)
        for v, t in zip(grp_ord["input_variant_id"], grp_ord["input_type"])
    ]
    _seen, _keep_idx = set(), []
    for _i, _k in enumerate(_keys):
        if _k not in _seen:
            _seen.add(_k)
            _keep_idx.append(_i)
    labels_distinct = [labels[_i] for _i in _keep_idx]
    m_distinct = max(len(labels_distinct) - 1, 0)
    _m_dist_distinct[m_distinct] += 1
    ent_d = _entropy_from_labels(labels_distinct)
    retained_distinct = bool(m_distinct >= MIN_M_ACCEPTED)
    if retained_distinct:
        _n_inc_dedup += 1

    # Accuracy + mapping_confidence from ORIGINAL-input row(s)
    orig = grp[grp["input_type"] == "original"]
    if len(orig) == 0:
        acc = 0
        map_conf = np.nan
    else:
        pred = _norm_cui(orig.iloc[0]["predicted_cui"])
        gold = _norm_cui(orig.iloc[0]["gold_cui_norm"] if "gold_cui_norm" in orig.columns else orig.iloc[0]["gold_cui"])
        acc = int(pred != UNASSIGNED and gold != UNASSIGNED and pred == gold)
        # Mean top-1 SapBERT/FAISS cosine of original-input output(s) → assigned CUI
        map_conf = float(pd.to_numeric(orig["confidence"], errors="coerce").mean())

    ent_rows.append({
        "instance_id": iid,
        "model_name": model,
        "domain": DOMAIN.get(model, "unknown"),
        "pair": PAIR.get(model),
        "normalised_entropy": ent["normalised_semantic_entropy_full"],
        "m_accepted": ent["m_accepted"],
        "accuracy": acc,
        "mapping_confidence": map_conf,
        "dominant_cui": ent["dominant_cluster_full"],
        "n_unassigned": ent["n_unassigned"],
        "all_unassigned": ent["all_unassigned"],
        "semantic_entropy": ent["semantic_entropy_full"],
        # Parallel de-duplicated view (not consumed by the RQ notebooks yet).
        "m_distinct": m_distinct,
        "normalised_entropy_dedup": ent_d["normalised_semantic_entropy_full"],
        "semantic_entropy_dedup": ent_d["semantic_entropy_full"],
        "dominant_cui_dedup": ent_d["dominant_cluster_full"],
        "n_unassigned_dedup": ent_d["n_unassigned"],
        "n_duplicate_variants": len(labels) - len(labels_distinct),
        "retained_m_accepted": True,
        "retained_m_distinct": retained_distinct,
    })

df_ent = pd.DataFrame(ent_rows)
_log(
    f"Inclusion m>=3: included(instance×model)={_n_inc:,}  excluded={_n_excl:,}"
)
_log(
    f"Parallel dedup view: retained_m_distinct={_n_inc_dedup:,} of {_n_inc:,} emitted rows "
    f"({_n_inc - _n_inc_dedup:,} would drop below m>={MIN_M_ACCEPTED} once byte-identical "
    f"variants are collapsed). Existing columns and the existing filter are unchanged."
)
print("m_distinct distribution (instance×model, before filter):")
print(pd.Series(_m_dist_distinct).sort_index().to_string())
print("m_accepted distribution (instance×model, before filter):")
for k in sorted(_m_dist):
    print(f"  m={k}: {_m_dist[k]:,}")

# Unique-instance coverage
_inst_m = (
    df_out[df_out["input_type"] == "perturbation"]
    .groupby("instance_id").size().rename("m")
)
_n_inst_keep = int((_inst_m >= MIN_M_ACCEPTED).sum())
_n_inst_drop = int((~(_inst_m >= MIN_M_ACCEPTED)).sum())
print(f"\nUnique instances with m>=3: included={_n_inst_keep:,}  excluded={_n_inst_drop:,}")
sys.stdout.flush()

assert len(df_ent) > 0, "No entropy rows after m>=3 filter"


entropy:   0%|          | 0/45352 [00:00<?, ?it/s]

entropy:   0%|          | 24/45352 [00:00<03:09, 238.68it/s]

entropy:   0%|          | 167/45352 [00:00<00:48, 937.29it/s]

entropy:   1%|          | 316/45352 [00:00<00:37, 1188.13it/s]

entropy:   1%|          | 457/45352 [00:00<00:35, 1274.57it/s]

...

entropy:  99%|█████████▉| 45094/45352 [00:31<00:00, 1457.56it/s]

entropy: 100%|█████████▉| 45240/45352 [00:31<00:00, 1428.69it/s]

entropy: 100%|██████████| 45352/45352 [00:31<00:00, 1433.62it/s]

[2026-07-31 12:31:11 UTC] Inclusion m>=3: included(instance×model)=45,352  excluded=0


m_accepted distribution (instance×model, before filter):
  m=3: 3,520
  m=4: 19,672
  m=5: 4,952
  m=6: 15,040
  m=7: 920
  m=8: 1,248

Unique instances with m>=3: included=5,669  excluded=0


## 5) Export + domain/pair summaries + asserts


In [6]:
# Write entropy_cadec.csv + prints + pair asserts
OUT_COLS = [
    "instance_id", "model_name", "domain", "pair",
    "normalised_entropy", "m_accepted", "accuracy", "mapping_confidence",
    "dominant_cui", "n_unassigned",
    # Parallel de-duplicated view, APPENDED so the existing columns and their order are
    # untouched. Downstream RQ notebooks keep reading normalised_entropy / m_accepted;
    # nothing consumes these yet. retained_m_distinct marks the rows that would survive
    # m>=3 once byte-identical input variants are collapsed.
    "m_distinct", "normalised_entropy_dedup", "semantic_entropy_dedup",
    "dominant_cui_dedup", "n_unassigned_dedup", "n_duplicate_variants",
    "retained_m_accepted", "retained_m_distinct",
]
df_save = df_ent[OUT_COLS].copy()
OUT_ENTROPY.parent.mkdir(parents=True, exist_ok=True)
df_save.to_csv(OUT_ENTROPY, index=False)
_log(f"Wrote {len(df_save):,} rows → {OUT_ENTROPY}")

print("\n===== Mean normalised_entropy per model =====")
_scored = df_ent.loc[~df_ent["all_unassigned"].astype(bool)].copy()
means = (
    _scored.groupby("model_name")["normalised_entropy"]
    .mean()
    .sort_values()
)
print(means.round(4).to_string())
print(f"\nOverall mean accuracy (original CUI match): {df_ent['accuracy'].mean():.4f}")
print(df_ent.groupby("model_name")["accuracy"].mean().round(4).to_string())
print("\n===== Mean mapping_confidence (original-input FAISS cosine) per model =====")
print(df_ent.groupby("model_name")["mapping_confidence"].mean().round(4).to_string())
print(
    f"mapping_confidence finite: {df_ent['mapping_confidence'].notna().mean():.1%} | "
    f"mean={df_ent['mapping_confidence'].mean():.4f}"
)
assert df_ent["mapping_confidence"].notna().any(), (
    "ASSERT FAIL: mapping_confidence all-NaN — check mapped confidence column."
)

print("\n===== Mean normalised_entropy per domain =====")
print(
    _scored.groupby("domain")["normalised_entropy"].mean().round(4).to_string()
)
sys.stdout.flush()

print("\n===== Matched-pair mean-H divergence asserts =====")
for bio_name, gen_name in MATCHED_PAIRS:
    hb = _scored.loc[_scored["model_name"] == bio_name, "normalised_entropy"].astype(float)
    hg = _scored.loc[_scored["model_name"] == gen_name, "normalised_entropy"].astype(float)
    if len(hb) == 0 or len(hg) == 0:
        print(f"SKIP assert {bio_name} vs {gen_name}: missing scored rows "
              f"(n_bio={len(hb)} n_gen={len(hg)})")
        continue
    mb, mg = float(hb.mean()), float(hg.mean())
    print(f"{bio_name}: mean_H={mb:.6f} (n={len(hb):,}) | "
          f"{gen_name}: mean_H={mg:.6f} (n={len(hg):,}) | Δ={mb-mg:+.6f}")
    assert not np.isclose(mb, mg, rtol=0, atol=1e-12), (
        f"IDENTICAL mean normalised_entropy bug signature: "
        f"{bio_name}={mb:.8f} vs {gen_name}={mg:.8f}"
    )
_log("ASSERT OK: matched-pair mean normalised_entropy values differ.")
print(f"\nOutput: {OUT_ENTROPY.resolve()}")


[2026-07-31 12:31:11 UTC] Wrote 45,352 rows → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/entropy_cadec.csv



===== Mean normalised_entropy per model =====
model_name
Llama3-OpenBioLLM-8B        0.0079
Meta-Llama-3-8B-Instruct    0.1354
Mistral-7B-Instruct-v0.1    0.1496
PubMedBERT                  0.1886
BioBERT                     0.1966
FLAN-T5-base                0.2028
BERT-base                   0.2265
BioMistral-7B               0.2323

Overall mean accuracy (original CUI match): 0.3370
model_name
BERT-base                   0.3156
BioBERT                     0.4103
BioMistral-7B               0.2436
FLAN-T5-base                0.2052
Llama3-OpenBioLLM-8B        0.4221
Meta-Llama-3-8B-Instruct    0.3362
Mistral-7B-Instruct-v0.1    0.3547
PubMedBERT                  0.4082

===== Mean mapping_confidence (original-input FAISS cosine) per model =====
model_name
BERT-base                   0.9973
BioBERT                     0.9977
BioMistral-7B               0.9822
FLAN-T5-base                0.9911
Llama3-OpenBioLLM-8B        0.9946
Meta-Llama-3-8B-Instruct    0.8401
Mistral-7B-Instruct-v


===== Matched-pair mean-H divergence asserts =====
BioBERT: mean_H=0.196593 (n=5,669) | BERT-base: mean_H=0.226521 (n=5,669) | Δ=-0.029928
BioMistral-7B: mean_H=0.232319 (n=5,659) | Mistral-7B-Instruct-v0.1: mean_H=0.149598 (n=5,633) | Δ=+0.082720


Llama3-OpenBioLLM-8B: mean_H=0.007950 (n=5,669) | Meta-Llama-3-8B-Instruct: mean_H=0.135355 (n=5,212) | Δ=-0.127405
[2026-07-31 12:31:11 UTC] ASSERT OK: matched-pair mean normalised_entropy values differ.



Output: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/entropy_cadec.csv
